<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## KKT: When a Bound Stops Improvement

**The best allowed setting can stop at a boundary before the error curve reaches its bottom.**

Keep the clock observations, squared-error objective, and exact calibration. Add an upper correction limit. One interactive graph compares the current candidate with the best allowed setting.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · The same clock, with one new limit

Choose initial correction $q$ (min) and rate correction $r$ (min/h). The four recorded errors are $-2,-1,-1,0$ min at times $0,1,2,3$ h. Write their components as $d_i,t_i$, for $i=1,\ldots,4$. The remaining errors $y_i$ and score are

$$
y_i=d_i+q+rt_i,\qquad \phi(q,r)=\sum_{i=1}^4 y_i^2.
$$

Using numerical values in these units, exact calibration at one hour gives $q+r=1$. Thus $r=1-q$, leaving the score

$$
p(q)=\phi(q,1-q)=0.5+6(q-1.5)^2.
$$

The bottom is at $q=1.5$. An upper limit $q\le q_{\max}=1$ excludes it. The best allowed setting is then $q=1$, $r=0$, with score $2$ min². Increasing $q$ would lower the error but break the requirement.


In the two-setting picture, calibration restricts candidates to a line. The upper bound keeps only the teal part of that line. The shaded half-plane alone is not enough: points off the line fail calibration.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-1_constrained_optimization_kkt/assets/calibration_feasible_set.svg" alt="Calibration requires q + r = 1; only the part with q at most 1 is feasible. The point (1, 0) is best, while (1.5, -0.5) is rejected." width="520" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · One graph: move the candidate or move the limit

The **candidate** slider changes $q$; calibration automatically sets $r=1-q$. The **limit** slider changes only $q_{\max}$. Orange marks an allowed candidate and gray marks a rejected one. The teal diamond marks the exact best allowed point:

$$
q^\star=\min(1.5,q_{\max}),\qquad r^\star=1-q^\star.
$$

With the limit fixed at $1$, moving the candidate from $0.75$ to $1$ lowers the score. Moving it to $1.5$ gives a still smaller score but is rejected. With the candidate fixed, moving the limit changes which points are allowed.

At the optimum, a bound is **active** if $q^\star=q_{\max}$. Otherwise it has unused room, called slack. The readout reports activity at the teal diamond, independently of the current candidate. The displayed range is a viewing window, not another constraint.


In [ ]:
import numpy as np

# Fixed observations; calibration remains exact.
observed_errors = np.array([-2., -1., -1., 0.])
observation_times = np.arange(4.)


def evaluate_clock(correction, upper_limit):
    rate = 1.0 - correction
    errors = observed_errors + correction + rate * observation_times
    return {"q": correction, "r": rate, "score": float(errors @ errors),
            "feasible": correction <= upper_limit + 1e-9}


def best_allowed(upper_limit):
    correction = min(1.5, upper_limit)
    result = evaluate_clock(correction, upper_limit)
    result["slack"] = upper_limit - correction
    result["active"] = abs(result["slack"]) < 1e-9
    result["slope"] = 12 * (correction - 1.5)
    result["mu"] = max(0.0, -result["slope"])
    return result


In [ ]:
import sys
import matplotlib

if sys.platform != "emscripten":
    try:
        matplotlib.use("widget", force=True)
    except (RuntimeError, ValueError):
        from matplotlib.backends import backend_registry
        backend_registry._clear()
        matplotlib.use("widget", force=True)
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
from IPython.display import display

BLUE, TEAL, ORANGE, GRAY, PURPLE = "#2878b5", "#168578", "#e78b24", "#707070", "#8854a5"
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11})

# Replace this notebook's previous canvas when this cell is rerun.
if "kkt_figure" in globals():
    plt.close(kkt_figure)
with plt.ioff():
    kkt_figure, axis = plt.subplots(figsize=(5.6, 5.7))
kkt_figure.subplots_adjust(left=0.16, right=0.96, top=0.90, bottom=0.49)
corrections = np.linspace(0.45, 2.3, 250)
axis.plot(corrections, 0.5 + 6 * (corrections - 1.5)**2, color=GRAY)
allowed_curve, = axis.plot([], [], color=TEAL, linewidth=2)
boundary = axis.axvline(1, color=TEAL, linestyle="--")
best_point, = axis.plot([], [], "D", color=TEAL, markersize=8, label="Best allowed")
candidate_point, = axis.plot([], [], "o", color=ORANGE, markersize=7, label="Candidate")
axis.set(xlabel="Correction q (min)", ylabel="Squared error (min²)",
         title="A better score must still satisfy the limit",
         xlim=(0.45, 2.3), ylim=(0, 7), xticks=[0.5, 1, 1.5, 2], yticks=[0, 2, 4, 6])
axis.grid(alpha=0.25)
axis.legend(loc="upper right", fontsize=10, framealpha=1)
status = kkt_figure.text(0.16, 0.35, "", va="top", linespacing=1.35)
q_slider = Slider(kkt_figure.add_axes([0.31, 0.12, 0.50, 0.03]), "Candidate q",
                  0.5, 2.25, valinit=0.75, valstep=0.05, valfmt="%.2f", color=BLUE)
limit_slider = Slider(kkt_figure.add_axes([0.31, 0.05, 0.50, 0.03]), "Limit",
                      0.5, 2.25, valinit=1.0, valstep=0.05, valfmt="%.2f", color=TEAL)


def update_kkt(_):
    limit = limit_slider.val
    candidate = evaluate_clock(q_slider.val, limit)
    best = best_allowed(limit)
    allowed = np.linspace(0.45, limit, 120)
    allowed_curve.set_data(allowed, 0.5 + 6 * (allowed - 1.5)**2)
    boundary.set_xdata([limit, limit])
    candidate_point.set_data([candidate["q"]], [candidate["score"]])
    candidate_point.set_color(ORANGE if candidate["feasible"] else GRAY)
    best_point.set_data([best["q"]], [best["score"]])
    verdict = "allowed" if candidate["feasible"] else "REJECTED"
    activity = "active" if best["active"] else "inactive"
    status.set_text(
        f"Candidate: r = {candidate['r']:.2f}; score = {candidate['score']:.3f}\n"
        f"Candidate is {verdict}\n"
        f"Best: q = {best['q']:.2f}; score = {best['score']:.3f}\n"
        f"Bound at best: {activity}; slack = {best['slack']:.2f}\n"
        f"At best: slope {best['slope']:.2f} + mu {best['mu']:.2f} = 0")
    kkt_figure._result = {"candidate": candidate, "best": best}
    kkt_figure.canvas.draw_idle()


kkt_figure._sliders = [q_slider, limit_slider]
for slider in kkt_figure._sliders:
    slider.label.set_x(-0.06)
    slider.valtext.set_x(1.06)
    slider.on_changed(update_kkt)
update_kkt(None)
if sys.platform == "emscripten":
    plt.show()
else:
    display(kkt_figure.canvas)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · The multiplier describes the blocked improvement

Along calibration, the slope is $p'(q)=12(q-1.5)$. At the best allowed point, the bound multiplier is $\mu=-p'(q^\star)$. It balances the remaining downhill slope; it is calculated, not chosen as a score weight.

| Limit $q_{\max}$ | Best $q^\star$ | Slope | $\mu$ | Bound at the optimum |
|:---|:---|:---|:---|:---|
| $1$ | $1$ | $-6$ | $6$ | Active: the limit blocks improvement. |
| $1.5$ | $1.5$ | $0$ | $0$ | Active, but already at the curve's bottom. |
| $2$ | $1.5$ | $0$ | $0$ | Inactive: room remains. |

**An active bound can have a zero multiplier.** At limit $2$, the boundary point $q=2$ is not optimal: its slope is $6$, and moving left is allowed and improves the score.


The strips compare the best setting with the limit, without changing the objective. A diamond touching the limit is active, even when its multiplier is zero. In the last strip, the gap is slack: 0.5 min of unused correction room.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-1_constrained_optimization_kkt/assets/bound_slack_cases.svg" alt="At limits 1 and 1.5 the best setting touches the bound, with multipliers 6 and 0. At limit 2 the best stays at 1.5, leaving slack 0.5 and multiplier 0." width="520" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · KKT gives four checks

The **Karush–Kuhn–Tucker (KKT) conditions** combine feasibility with slope balance. Because $r=1-q$ already enforces calibration, the graph uses these checks:

| Check | Equation | Meaning |
|:---|:---|:---|
| Feasibility | $q\le q_{\max}$ | The candidate is allowed. |
| Multiplier sign | $\mu\ge0$ | An upper bound can block movement to the right. |
| Complementarity | $\mu(q-q_{\max})=0$ | Unused room makes the bound multiplier zero. |
| Stationarity | $p'(q)+\mu=0$ | The slope and bound contribution balance. |

At limit $1$, $q=1$ and $\mu=6$ satisfy all four: $1\le1$, $6\ge0$, $6(1-1)=0$, and $-6+6=0$.

The equal-length arrows show the signed terms in stationarity. The negative objective slope and positive bound term cancel. These arrows represent equation terms, not proposed changes in the clock setting.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-1_constrained_optimization_kkt/assets/stationarity_balance.svg" alt="At correction q = 1, the objective slope term is minus 6 and the bound term is plus 6; their sum is zero." width="520" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>


For both original settings, write equality residual $h=q+r-1$ and inequality residual $g=q-q_{\max}$. With equality multiplier $\nu$, the Lagrangian is $L=\phi+\nu h+\mu g$. Stationarity becomes

$$
\frac{\partial\phi}{\partial q}+\nu+\mu=0,
\qquad \frac{\partial\phi}{\partial r}+\nu=0.
$$

At $(q,r)=(1,0)$, the two objective slopes are $0$ and $6$. Multipliers $\nu=-6$, $\mu=6$ balance them: $0-6+6=0$ and $6-6=0$. The equality multiplier has no sign restriction.

For this strictly convex quadratic with linear constraints, KKT certifies the unique global minimum. In general, KKT is necessary at a smooth local minimum only under suitable constraint regularity assumptions, and satisfying it need not prove a minimum. KKT checks a candidate; it is not a search algorithm.
